In [28]:
# importing the Python Libraries
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [29]:
# load from own EDA export, 11,140-row cleaned data
cleaned_customer_transaction_data = pd.read_csv(r'../Finlora_Dataset/artifacts/EDA_Data.csv')

print(cleaned_customer_transaction_data.shape)

(11140, 30)


## Threshold-Based Risk Features

Created binary risk indicators for signals confirmed independently predictive during EDA: late night hours (3am-8am), elevated IP risk score (above 0.8), low device trust score (below 0.3), new and very new account age, and transaction velocity spikes. Verified the 30/90-day account age split against the live data: fraud rate peaks at 43.8% in the 30-90 day window versus 35.6% under 30 days, so `new_account` is the stronger of the two flags despite `very_new_account` sounding more extreme. `kyc_tier` and `location_mismatch` are handled at the encoding stage instead of as manual thresholds, since both are already categorical or binary.

In [30]:
# late night window matches the 3am-8am fraud spike confirmed in EDA
cleaned_customer_transaction_data['late_night_hours'] = (
    (cleaned_customer_transaction_data['hour'] >= 3) &
    (cleaned_customer_transaction_data['hour'] <= 8)
)

# elevated ip risk score, matches the cliff at 0.8 confirmed in EDA
cleaned_customer_transaction_data['high_ip_risk'] = (
    cleaned_customer_transaction_data['ip_risk_score'] > 0.8
)

# low device trust score, matches the cliff at 0.3 confirmed in EDA
cleaned_customer_transaction_data['low_device_trust'] = (
    cleaned_customer_transaction_data['device_trust_score'] < 0.3
)

# 30/90 boundary verified against the live data: 43.8% fraud rate in this
# window, the strongest of the two account-age flags
cleaned_customer_transaction_data['new_account'] = (
    (cleaned_customer_transaction_data['account_age_days'] >= 30) &
    (cleaned_customer_transaction_data['account_age_days'] <= 90)
)

# note: fraud rate peaks in the 30-90 day window (43.8%), not under 30 days
# (35.6%). very_new_account is real signal, just not the stronger of the two
cleaned_customer_transaction_data['very_new_account'] = (
    cleaned_customer_transaction_data['account_age_days'] < 30
)

# velocity spike, matches the jump at 3+ transactions per hour confirmed in EDA
cleaned_customer_transaction_data['velocity_spike'] = (
    cleaned_customer_transaction_data['txn_velocity_1h'] >= 3
)

amount_usd deliberately excluded from threshold flags here. EDA cross-tab confirmed its fraud-rate spike is confounded with device_trust_score and ip_risk_score, not an independent effect, kept as a raw feature instead.

In [31]:
# Combining the threshold flags into a single feature dataframe for review
high_risk_signal_features = cleaned_customer_transaction_data[[
    'late_night_hours',
    'high_ip_risk',
    'low_device_trust',
    'new_account',
    'very_new_account',
    'velocity_spike'
]].astype(int)

high_risk_signal_features.head()

,late_night_hours,high_ip_risk,low_device_trust,new_account,very_new_account,velocity_spike
0,0,0,0,0,0,0
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0


In [32]:
# checking the features in our dataset
list(cleaned_customer_transaction_data.columns)

['transaction_id',
 'customer_id',
 'timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'exchange_rate_src_to_dest',
 'device_id',
 'new_device',
 'ip_address',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'chargeback_history_count',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud',
 'hour',
 'day_of_week',
 'is_weekend',
 'month',
 'late_night_hours',
 'high_ip_risk',
 'low_device_trust',
 'new_account',
 'very_new_account',
 'velocity_spike']

In [33]:
# Threshold flags changed to int 
threshold_columns = [
    'late_night_hours',
    'high_ip_risk',
    'low_device_trust',
    'new_account',
    'very_new_account',
    'velocity_spike'
]
cleaned_customer_transaction_data[threshold_columns] = (
    cleaned_customer_transaction_data[threshold_columns].astype(int)
)


`chargeback_history_count` is kept as a feature here. A fraud-rate breakdown shows a massive split: customers with 0 recorded chargebacks have a 5.4% fraud rate (n=10,699), 1 chargeback jumps to 93.0% (n=330), and 2 chargebacks hits 99.1% (n=111). All three sample sizes are well above the reliability threshold used throughout this project (n=30).

Checked for leakage before trusting it: if the count included a chargeback filed against the very transaction being classified, almost every fraud row would show a count above zero. Instead, 58.0% of all fraud transactions have zero chargeback history, only 42.0% carry any history at all. That rules out the column simply restating the outcome. What it captures is a smaller population of repeat offenders who get flagged almost every time, sitting on top of a much larger population of first-time fraud with no prior history. Verdict: real predictive signal, not leakage, kept as a feature with no reservation.

In [36]:
# identifiers carry no signal on their own, timestamp's signal is already in
# hour, day_of_week, is_weekend. month and the fx rate had no EDA signal
drop_columns = [
    'transaction_id',
    'customer_id',
    'device_id',
    'ip_address',
    'timestamp',
    'month',
    'exchange_rate_src_to_dest'
]

In [37]:
# combined drop, keeps chargeback_history_count as a feature given its confirmed signal
action_data = cleaned_customer_transaction_data.drop(columns=drop_columns)
action_data.shape

(11140, 29)

In [38]:
# categorical features go through one hot encoding
categorical_features = action_data.select_dtypes(include=['object', 'bool']).columns
categorical_features

Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'new_device', 'ip_country', 'location_mismatch', 'kyc_tier'],
      dtype='object')

In [39]:
# numerical features go through imputation and scaling, is_fraud is the target not a feature
numerical_features = action_data.select_dtypes(include=['int', 'float']).columns.drop('is_fraud')
numerical_features

Index(['amount_src', 'amount_usd', 'fee', 'ip_risk_score', 'account_age_days',
       'device_trust_score', 'chargeback_history_count', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'hour',
       'day_of_week', 'is_weekend', 'late_night_hours', 'high_ip_risk',
       'low_device_trust', 'new_account', 'very_new_account',
       'velocity_spike'],
      dtype='object')

customer_id has only 1,315 unique values across 11,140 rows, each customer appears roughly 8.5 times on average. This means a plain random train_test_split would likely split one customer's transactions across both train and test. Since account-level fields like chargeback_history_count stay close to identical across a customer's own rows, the model could partly memorize a customer's risk profile from train and score artificially well on test rows from that same customer. Needs a customer-grouped split (GroupShuffleSplit keyed on customer_id) instead of a plain train_test_split when that step is reached.

In [40]:
# confirm the final column set going into modelling
action_data.columns

Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'amount_src', 'amount_usd', 'fee', 'new_device', 'ip_country',
       'location_mismatch', 'ip_risk_score', 'kyc_tier', 'account_age_days',
       'device_trust_score', 'chargeback_history_count', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'hour', 'day_of_week', 'is_weekend', 'late_night_hours', 'high_ip_risk',
       'low_device_trust', 'new_account', 'very_new_account',
       'velocity_spike'],
      dtype='object')

In [41]:
# check dtypes and non-null counts 
action_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11140 entries, 0 to 11139
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   home_country              11140 non-null  object 
 1   source_currency           11140 non-null  object 
 2   dest_currency             11140 non-null  object 
 3   channel                   11140 non-null  object 
 4   amount_src                11140 non-null  float64
 5   amount_usd                11140 non-null  float64
 6   fee                       11140 non-null  float64
 7   new_device                11140 non-null  bool   
 8   ip_country                11140 non-null  object 
 9   location_mismatch         11140 non-null  bool   
 10  ip_risk_score             11140 non-null  float64
 11  kyc_tier                  11140 non-null  object 
 12  account_age_days          11140 non-null  int64  
 13  device_trust_score        11140 non-null  float64
 14  charge

In [42]:
# save the finalized feature set 
action_data.to_csv("../Finlora_Dataset/artifacts/Engineered_Data.csv", index=False)